# mlX 2.0 Regression Challenge: Assignment Notebook

## Objective
Predict song popularity (`target`, 0-100) and generate a valid submission (`id`, `target`) while comparing at least two regression approaches.

## Approaches Used
1. **Ridge Regression (Linear Baseline)**
- Why chosen: fast, stable, and interpretable baseline for mixed tabular features after one-hot encoding.
- How it works: learns a linear relationship and uses L2 regularization to reduce overfitting.
- Drawbacks: cannot model complex non-linear interactions as effectively.

2. **Random Forest Regressor (Non-linear Ensemble)**
- Why chosen: strong tabular performance and ability to model non-linear patterns and interactions.
- How it works: averages many decision trees trained on bootstrapped data subsets.
- Drawbacks: heavier computation, lower interpretability, and can overfit if not tuned.

## Evaluation Metrics
- RMSE (competition metric)
- MAE
- R2
- MAPE (with safe denominator handling)

In [ ]:
# This notebook keeps Kaggle competition structure and output format.
# It trains two regression approaches and selects the better one by holdout RMSE.

import os
import numpy as np
import pandas as pd

from sklearn.model_selection import KFold, train_test_split, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Kaggle input listing (works on Kaggle runtime).
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

RANDOM_STATE = 42

def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def mape_safe(y_true, y_pred, eps=1e-8):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    denom = np.maximum(np.abs(y_true), eps)
    return float(np.mean(np.abs((y_true - y_pred) / denom)) * 100.0)

def locate_data_files():
    # Primary path for this Kaggle competition.
    kaggle_train = "/kaggle/input/competitions/mlx-2-0-regression/train.csv"
    kaggle_test = "/kaggle/input/competitions/mlx-2-0-regression/test.csv"
    if os.path.exists(kaggle_train) and os.path.exists(kaggle_test):
        return kaggle_train, kaggle_test

    # Local fallback for offline testing.
    candidates = [
        ("train.csv", "test.csv"),
        ("./data/train.csv", "./data/test.csv"),
    ]
    for train_path, test_path in candidates:
        if os.path.exists(train_path) and os.path.exists(test_path):
            return train_path, test_path

    raise FileNotFoundError(
        "Could not find train.csv and test.csv in Kaggle input, working directory, or ./data/."
    )

/kaggle/input/competitions/mlx-2-0-regression/sample_submission.csv
/kaggle/input/competitions/mlx-2-0-regression/train.csv
/kaggle/input/competitions/mlx-2-0-regression/test.csv


In [ ]:
# Load competition files
train_path, test_path = locate_data_files()
print("Using:", train_path, "and", test_path)

train_data = pd.read_csv(train_path)
test_data = pd.read_csv(test_path)

print("Train shape:", train_data.shape)
print("Test shape:", test_data.shape)
display(train_data.head())
print("Train columns:", list(train_data.columns))

In [ ]:
missing_pct = (train_data.isnull().mean() * 100).sort_values(ascending=False)
print("Top columns by missing percentage:")
display(missing_pct.head(15))

target_desc = train_data["target"].describe()
print("Target summary:")
display(target_desc)

Index(['id', 'emotional_charge_2', 'groove_efficiency_1', 'beat_frequency_1',
       'organic_texture_2', 'composition_label_0', 'harmonic_scale_1',
       'intensity_index_0', 'duration_ms_0', 'album_name_length',
       'beat_frequency_0', 'beat_frequency_2', 'artist_count',
       'composition_label_1', 'publication_timestamp', 'weekday_of_release',
       'album_component_count', 'emotional_charge_1', 'emotional_charge_0',
       'tonal_mode_2', 'key_variety', 'performance_authenticity_2',
       'performance_authenticity_0', 'season_of_release', 'time_signature_1',
       'duration_ms_2', 'lunar_phase', 'instrumental_density_2',
       'organic_texture_0', 'creator_collective', 'vocal_presence_2',
       'tonal_mode_1', 'vocal_presence_1', 'vocal_presence_0',
       'intensity_index_1', 'organic_immersion_0', 'tonal_mode_0',
       'groove_efficiency_2', 'instrumental_density_1', 'organic_immersion_2',
       'duration_consistency', 'composition_label_2', 'organic_texture_1',
    

In [ ]:
X = train_data.drop(columns=["target"])
y = train_data["target"]

num_cols = X.select_dtypes(include=["number"]).columns.tolist()
cat_cols = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

print(f"Numeric features: {len(num_cols)}")
print(f"Categorical features: {len(cat_cols)}")

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)
print("Train split:", X_train.shape, "Valid split:", X_valid.shape)

tempo_volatility       20.348883
beat_frequency_0       18.757469
album_name_length      18.444679
duration_ms_1          17.341536
creator_collective     16.916216
                         ...    
groove_efficiency_1     0.293021
organic_immersion_1     0.224496
vocal_presence_0        0.178865
id                      0.000000
target                  0.000000
Length: 62, dtype: float64

In [ ]:
# Runtime controls
FAST_MODE = False  # Set False for final thorough run before submission
CV_FOLDS = 3 if FAST_MODE else 5
RF_TREES = 180 if FAST_MODE else 500

# Approach 1 (Linear baseline): Ridge Regression
linear_preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]),
            num_cols,
        ),
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore")),
            ]),
            cat_cols,
        ),
    ]
)

ridge_model = Pipeline([
    ("preprocessor", linear_preprocessor),
    ("regressor", Ridge(alpha=2.0)),
])

# Approach 2 (Non-linear ensemble): Random Forest
tree_preprocessor = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), num_cols),
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=10)),
            ]),
            cat_cols,
        ),
    ]
)

rf_model = Pipeline([
    ("preprocessor", tree_preprocessor),
    (
        "regressor",
        RandomForestRegressor(
            n_estimators=RF_TREES,
            max_depth=14 if FAST_MODE else 18,
            min_samples_leaf=2,
            n_jobs=-1,
            random_state=RANDOM_STATE,
        ),
    ),
])

models = {
    "Ridge": ridge_model,
    "RandomForest": rf_model,
}

print(f"FAST_MODE={FAST_MODE} | CV_FOLDS={CV_FOLDS} | RF_TREES={RF_TREES}")

## Model Building Notes

### Approach 1: Ridge Regression
- Acts as a strong baseline for regression on one-hot encoded mixed features.
- Regularization (`alpha`) controls coefficient magnitude and improves generalization.
- Limitation: mainly captures linear effects.

### Approach 2: Random Forest
- Captures non-linear interactions among audio and metadata features.
- Robust to outliers and feature scaling issues in numeric variables.
- Limitation: larger training/inference cost and less transparent decision logic.

In [ ]:
cv = KFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)

results = []
fitted_models = {}

for name, model in models.items():
    scores = cross_validate(
        model,
        X,
        y,
        cv=cv,
        scoring={
            "rmse": "neg_root_mean_squared_error",
            "mae": "neg_mean_absolute_error",
            "r2": "r2",
        },
        n_jobs=-1,
    )

    # Fit once on holdout split and keep the fitted model for fast inference.
    model.fit(X_train, y_train)
    fitted_models[name] = model
    valid_pred = model.predict(X_valid)

    row = {
        "model": name,
        "cv_rmse_mean": -scores["test_rmse"].mean(),
        "cv_rmse_std": scores["test_rmse"].std(),
        "cv_mae_mean": -scores["test_mae"].mean(),
        "cv_r2_mean": scores["test_r2"].mean(),
        "holdout_rmse": rmse(y_valid, valid_pred),
        "holdout_mae": mean_absolute_error(y_valid, valid_pred),
        "holdout_r2": r2_score(y_valid, valid_pred),
        "holdout_mape": mape_safe(y_valid, valid_pred),
    }
    results.append(row)

results_df = pd.DataFrame(results).sort_values("holdout_rmse").reset_index(drop=True)
display(results_df)

best_model_name = results_df.loc[0, "model"]
best_model = fitted_models[best_model_name]
print("Best model by holdout RMSE:", best_model_name)

[Pipeline] ........... (step 1 of 1) Processing imputer, total=   0.4s
[ColumnTransformer] ........... (1 of 2) Processing num, total=   0.4s
[Pipeline] ........... (step 1 of 2) Processing imputer, total=   0.1s
[Pipeline] ............ (step 2 of 2) Processing onehot, total=   1.1s
[ColumnTransformer] ........... (2 of 2) Processing cat, total=   1.2s
[Pipeline] ...... (step 1 of 2) Processing preprocessor, total=   1.9s
[Pipeline] ......... (step 2 of 2) Processing regressor, total= 5.9min


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))],
                                                           verbose=True),
                                                  Index(['id', 'emotional_charge_2', 'groove_efficiency_1', 'beat_frequency_1',
       'organic_texture_2', 'harmonic_scale_1', 'intensity_index_0',
       'duration_ms_0', 'album_name_length', 'beat_frequency_0',
       'beat_frequency...
                                                                                 min_frequency=10))],
                                                           verbose=True),
                                                  Index(['composition_label_0', 'composition_label_1', 'publication_timestamp',
       'weekday_of_release', 'season_of_release', 'lunar_phase',
       'creator_collective', 'composition_label_2', 'track_identifier'],
      dtype='object'))],
                                   verbose=True)),
                ('regressor',
                 RandomForestRegressor(max_depth=15, min_samples_leaf=2,
                                       n_jobs=-1, random_state=42))],
         verbose=True)

## Metric Discussion and Comparison

- **RMSE** is the official leaderboard metric and penalizes larger misses more strongly.
- **MAE** gives average absolute error in popularity points and is easier to interpret.
- **R2** reports how much target variance is explained.
- **MAPE** gives relative error in percentage terms; it is less reliable when true values are very close to zero.

Use the results table to compare both approaches. The model with lower holdout RMSE is selected for final submission.

In [ ]:
# Build submission from selected best model
RETRAIN_ON_FULL_DATA = False  # Set True for final Kaggle submission if you want max accuracy

if RETRAIN_ON_FULL_DATA:
    final_model = models[best_model_name]
    final_model.fit(X, y)
else:
    final_model = best_model  # already fitted on X_train in previous cell

test_preds = final_model.predict(test_data)

submission = pd.DataFrame({
    "id": test_data["id"],
    "target": np.clip(test_preds, 0, 100),
})
submission.to_csv("submission.csv", index=False)

print("Saved submission.csv")
print("Retrained on full data:", RETRAIN_ON_FULL_DATA)
display(submission.head())

print("\nShort metric discussion:")
print("- RMSE: competition metric; penalizes large errors strongly.")
print("- MAE: average absolute error; easier to interpret in popularity points.")
print("- R2: variance explained by model; useful for relative goodness of fit.")
print("- MAPE: relative percent error; can be unstable near zero targets.")

print("\nApproach comparison summary:")
for _, r in results_df.iterrows():
    print(
        f"{r['model']}: holdout RMSE={r['holdout_rmse']:.4f}, "
        f"MAE={r['holdout_mae']:.4f}, R2={r['holdout_r2']:.4f}, MAPE={r['holdout_mape']:.2f}%"
    )

      id     target
0  25174  70.982260
1  38453  64.954460
2  29013  57.115698
3  57463  75.760270
4  51264  39.552117
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41074 entries, 0 to 41073
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   id      41074 non-null  int64  
 1   target  41074 non-null  float64
dtypes: float64(1), int64(1)
memory usage: 641.9 KB
